# Citation Drift — Colab Environment Check

**Stop after this notebook.** Do not install AlignScore or download data/model weights yet.

1. In Colab select **Runtime → Change runtime type → T4 GPU** if available.
2. Run the cells in order.
3. Share the complete output with the research software engineer.

In [ ]:
import os, sys, platform, subprocess, json, datetime, importlib.util
print("UTC:", datetime.datetime.now(datetime.timezone.utc).isoformat())
print("Python:", sys.version)
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("Architecture:", platform.machine())
print("Working directory:", os.getcwd())

In [ ]:
# NVIDIA / CUDA diagnostics. A missing nvidia-smi can mean CPU runtime or unavailable GPU.
try:
    p = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=20)
    print("nvidia-smi return code:", p.returncode)
    print(p.stdout if p.stdout else p.stderr)
except Exception as e:
    print("nvidia-smi diagnostic:", repr(e))

In [ ]:
# PyTorch diagnostics: do not reinstall torch before reviewing this output.
try:
    import torch
    print("torch:", torch.__version__)
    print("torch CUDA build:", torch.version.cuda)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU count:", torch.cuda.device_count())
        for i in range(torch.cuda.device_count()):
            print(f"GPU {i}:", torch.cuda.get_device_name(i))
        x = torch.tensor([1.0, 2.0], device="cuda")
        print("CUDA tensor test:", (x * 2).sum().item(), "device:", x.device)
    else:
        print("CUDA tensor test: SKIPPED (no CUDA GPU exposed)")
    print("cuDNN:", torch.backends.cudnn.version())
except Exception as e:
    print("PyTorch diagnostic FAILED:", repr(e))

In [ ]:
# Check key preinstalled package versions without changing the runtime.
from importlib.metadata import version, PackageNotFoundError
packages = ["numpy", "pandas", "scikit-learn", "transformers",
            "pytorch-lightning", "spacy", "nltk", "datasets",
            "scipy", "protobuf", "rank-bm25", "pypdf"]
for pkg in packages:
    try:
        print(f"{pkg}=={version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg}: NOT INSTALLED")

In [ ]:
# Tiny CPU tensor and scientific-library import test.
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
a = np.array([1, 2, 3])
print("NumPy sum:", int(a.sum()))
print("Pandas version:", pd.__version__)
print("sklearn F1 sanity:", f1_score([0, 1], [0, 1]))

In [ ]:
# Save a structured environment log in the Colab VM.
from pathlib import Path
import importlib.metadata as md
import torch

out = {
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "architecture": platform.machine(),
    "torch": torch.__version__,
    "torch_cuda_build": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "packages": {}
}
for pkg in ["numpy", "pandas", "scikit-learn", "transformers", "pytorch-lightning",
            "spacy", "nltk", "datasets", "scipy", "protobuf", "rank-bm25", "pypdf"]:
    try:
        out["packages"][pkg] = md.version(pkg)
    except md.PackageNotFoundError:
        out["packages"][pkg] = None

Path("/content/environment_verification.json").write_text(json.dumps(out, indent=2))
print(json.dumps(out, indent=2))
print("\nSaved: /content/environment_verification.json")

## Expected output type

- Runtime metadata and architecture strings.
- `nvidia-smi` report or a clear unavailable-GPU message.
- PyTorch version, CUDA build, GPU availability and (if possible) a successful CUDA tensor result.
- Installed package versions or `NOT INSTALLED`.
- NumPy/Pandas/scikit-learn sanity-test values.
- A JSON file at `/content/environment_verification.json`.

## Failure diagnostics

- `nvidia-smi: not found` or no GPU: confirm Runtime → Change runtime type → GPU, then reconnect.
- `torch.cuda.is_available() == False` despite GPU runtime: share `nvidia-smi`, torch version and `torch.version.cuda`; do not reinstall torch yet.
- Import errors: share full traceback and Python version.
- NumPy/sklearn binary ABI errors: likely preinstalled package mismatch; do not randomly upgrade packages.
- Runtime reset/disconnect: rerun from the first cell; `/content` files may be lost.
- No fixed GPU model or memory size is guaranteed by Colab; report what the runtime actually provides.

**Stop here and send me all cell outputs.**